# Suite P3CONF — the Python corpus

The conformance files this implementation writes for Python (`python -m mbse.Programs.Conformance.write`), as the user
decided for the port: each source's frame, the `References.OfFrame` of the transaction its parse gives, and the text
its module prints. The whole tree is values: the frame binds only the workspace (`ws`) and the module (`module`), and
its one assignment gives the module its `body`, every statement nested in it as one value. Every implementation writes
the same bytes; the TypeScript implementation joins with its stage.

In [ ]:
import ast
import json
from pathlib import Path

from mbse.Programs import Python3
from mbse.Programs.Conformance import write
from mbse.Schemas.Framework import Comparison, Plain

CORPUS = write.DEFAULT
SOURCES = write.ROOT / "sources"

## P3CONF-01 · The corpus files are current

In [ ]:
rendered = write.render()
# every language ported so far: TS5CONF checks TypeScript's, CCONF C's
assert sorted(rendered) == ["c23.c", "c23.json", "python314.json", "python314.py", "typescript59-jsx.json",
                           "typescript59-jsx.tsx", "typescript59.json", "typescript59.ts"]
for name, text in rendered.items():
    assert (CORPUS / name).read_text(encoding="utf-8") == text, f"{name} is stale: run python -m mbse.Programs.Conformance.write"


## P3CONF-02 · A frame read back compares equal, and is written again as it was

In [ ]:
text = (CORPUS / "python314.json").read_text(encoding="utf-8")
store = Python3.store()
schema = store.registered("References.OfFrame")
fresh = Python3.store()
mine = json.loads(write.frame(fresh, Python3.Parse.parse((SOURCES / "python314.py").read_text(encoding="utf-8"), "python314.py")))
read = Plain.FromPlain(store).Value(schema, json.loads(text))
again = Plain.FromPlain(store).Value(schema, mine)
assert Comparison.OfObject(schema, read).compare(Comparison.OfObject(schema, again)) == 0
assert json.dumps(Plain.ToPlain(store).Value(schema, read), indent=2, ensure_ascii=False) + "\n" == text
frame = json.loads(text)
assert set(frame) == {"binds", "assigns"} and set(frame["binds"]) == {"ws", "module"}
(assign,) = frame["assigns"]  # the module, given its whole tree as one nested value
assert assign["object"] == frame["binds"]["module"] and set(assign["properties"]) == {"body"}

## P3CONF-03 · The printed corpus means what its source does, and prints as it is

In [ ]:
source = (SOURCES / "python314.py").read_text(encoding="utf-8")
printed = (CORPUS / "python314.py").read_text(encoding="utf-8")
assert ast.dump(ast.parse(printed)) == ast.dump(ast.parse(source))
store = Python3.store()
store.transaction(Python3.Parse.parse(printed, "printed.py"))
assert Python3.Print.print_module(store, "printed.py") == printed
